# Tari'ak Traffic Analysis, 00 — Dataset Feasibility

This notebook evaluates the Tari'ak Lebanon traffic dataset for a focused AI and machine-learning project. It establishes the observation grain, verifies the raw schema and data quality, identifies the temporal and geographic coverage, and assesses which analytical and predictive tasks the data can legitimately support.

Model selection and target definition will follow the evidence from the dataset rather than being assumed in advance.

## 1. Setup and raw-file location

The source file is comma-separated despite its `.txt` extension. Because it is large, the initial inspection reads a small sample to confirm parsing and row structure before loading the full dataset.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
RAW_PATH = RAW_DIR / 'velocities.txt'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

print(f'Raw file exists: {RAW_PATH.exists()}')
print(f'Raw file size: {RAW_PATH.stat().st_size / 1024**2:,.1f} MB')

Raw file exists: True
Raw file size: 515.2 MB


## 2. First raw sample

This initial inspection checks the field names, inferred data types, example values, and whether the parser identifies the expected six fields per row.

In [2]:
sample_df = pd.read_csv(
    RAW_PATH,
    nrows=10_000,
    sep=',',
    skipinitialspace=True,
    low_memory=False
)

sample_df.columns = (
    sample_df.columns
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print('Shape:', sample_df.shape)
print('Columns:', sample_df.columns.tolist())
print('\nDtypes:')
display(sample_df.dtypes.to_frame('dtype'))
print('\nFirst rows:')
display(sample_df.head())

Shape: (10000, 6)
Columns: ['Date', 'Time', 'Coordinate (Lon, Lat)', 'Course', 'Velocity', 'OSM ID']

Dtypes:


,dtype
Date,object
Time,object
"Coordinate (Lon, Lat)",object
Course,float64
Velocity,float64
OSM ID,object



First rows:


,Date,Time,"Coordinate (Lon, Lat)",Course,Velocity,OSM ID
0,2015-03-20,00:00:48,"35.49234430,33.88014290",111.975845,36.803894,way/262256295
1,2015-03-20,00:01:34,"35.65125300,34.01199400",141.987791,50.580000,way/227026194
2,2015-03-20,00:01:36,"35.49638720,33.87891840",110.530757,12.359205,way/211496405
3,2015-03-20,00:02:17,"35.49492840,33.87586950",201.755776,32.005948,way/179130339
4,2015-03-20,00:02:55,"35.49352200,33.85363600",175.191792,111.491996,way/178472583


In [3]:
print('Missing values in sample:')
display(sample_df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print('Unique values in identifier-like fields:')
print('OSM ID:', sample_df['OSM ID'].nunique())
print('Coordinates:', sample_df['Coordinate (Lon, Lat)'].nunique())

print('Numeric summary:')
display(sample_df[['Course', 'Velocity']].describe())

Missing values in sample:


,missing_count
Date,0
Time,0
"Coordinate (Lon, Lat)",0
Course,0
Velocity,0
OSM ID,0


Unique values in identifier-like fields:
OSM ID: 2030
Coordinates: 10000
Numeric summary:


,Course,Velocity
count,10000.000000,10000.000000
mean,168.292409,42.468022
std,103.764659,28.118632
min,0.049416,0.001629
25%,72.546070,20.124000
50%,182.193214,39.389092
75%,249.673399,62.997791
max,359.972235,119.916004


### Initial observations

The first 10,000 rows load into six expected fields with no missing values in the sample. `Course` spans approximately 0–360, which is consistent with a directional angle. `Velocity` is non-negative in the sample and ranges from almost zero to about 119.9, but the unit and measurement semantics still need to be verified before interpreting these values as a real-world speed.

The 2,030 unique `OSM ID` values show that the file contains repeated observations associated with road segments rather than one record per road segment. The sample has 10,000 unique coordinate strings, so coordinates should be parsed and checked independently rather than treated as identifiers.

These results establish that the file is structurally usable for deeper inspection. They do not yet establish that it supports a legitimate prediction task: we still need to understand the time coverage, sampling frequency, spatial distribution, repeated road-segment observations, duplicate rows, and the relationship between velocity and the available features.

## 3. Inspection plan

The next inspection will proceed in four stages:

1. Parse `Date` and `Time` into a timestamp and measure the full temporal coverage.
2. Split the coordinate field into numeric longitude and latitude, then check geographic ranges and invalid values.
3. Load the full file in a controlled way and quantify missing rows, exact duplicates, repeated road segments, and observations per segment.
4. Examine velocity distributions across time and location to determine whether the data can support a useful prediction or estimation problem.

Only after these checks will the project define a target variable, baseline, and evaluation strategy.

## 4. Timestamp and coordinate validation

The raw file stores date, time, and both coordinates inside text fields. These fields must be converted into usable numeric and temporal variables before we can assess coverage, sampling behavior, or spatial patterns.

In [4]:
inspection_df = sample_df.copy()

inspection_df['timestamp'] = pd.to_datetime(
    inspection_df['Date'].astype(str) + ' ' + inspection_df['Time'].astype(str),
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

coordinate_parts = inspection_df['Coordinate (Lon, Lat)'].astype('string').str.extract(r'^\s*([^,]+),\s*([^,]+)\s*$')
inspection_df['longitude'] = pd.to_numeric(coordinate_parts[0].str.strip(), errors='coerce')
inspection_df['latitude'] = pd.to_numeric(coordinate_parts[1].str.strip(), errors='coerce')

print('Timestamp parsing failures:', inspection_df['timestamp'].isna().sum())
print('Coordinate parsing failures:', inspection_df[['longitude', 'latitude']].isna().any(axis=1).sum())
print('Timestamp range in sample:', inspection_df['timestamp'].min(), 'to', inspection_df['timestamp'].max())
print('Longitude range:', inspection_df['longitude'].min(), 'to', inspection_df['longitude'].max())
print('Latitude range:', inspection_df['latitude'].min(), 'to', inspection_df['latitude'].max())

display(inspection_df[['timestamp', 'longitude', 'latitude', 'Course', 'Velocity']].head())

Timestamp parsing failures: 0
Coordinate parsing failures: 0
Timestamp range in sample: 2015-03-20 00:00:48 to 2015-03-21 14:30:02
Longitude range: 35.139195 to 36.423145
Latitude range: 33.0811812 to 34.6231104


,timestamp,longitude,latitude,Course,Velocity
0,2015-03-20 00:00:48,35.492344,33.880143,111.975845,36.803894
1,2015-03-20 00:01:34,35.651253,34.011994,141.987791,50.580000
2,2015-03-20 00:01:36,35.496387,33.878918,110.530757,12.359205
3,2015-03-20 00:02:17,35.494928,33.87587,201.755776,32.005948
4,2015-03-20 00:02:55,35.493522,33.853636,175.191792,111.491996


In [5]:
sorted_timestamps = inspection_df['timestamp'].dropna().sort_values()
time_gaps = sorted_timestamps.diff().dt.total_seconds().dropna()

print('Unique timestamps:', inspection_df['timestamp'].nunique())
print('Duplicate complete rows in sample:', inspection_df.duplicated().sum())
print('Non-positive coordinates:', ((inspection_df['longitude'] == 0) | (inspection_df['latitude'] == 0)).sum())
print('Out-of-range coordinates:', ((inspection_df['longitude'].abs() > 180) | (inspection_df['latitude'].abs() > 90)).sum())

print('Sample timestamp gap summary in seconds:')
display(time_gaps.describe())

print('Observations by date in sample:')
display(inspection_df['timestamp'].dt.date.value_counts().sort_index().head(20))

Unique timestamps: 9479
Duplicate complete rows in sample: 0
Non-positive coordinates: 0
Out-of-range coordinates: 0
Sample timestamp gap summary in seconds:


count    9999.000000
mean       13.856786
std        53.510967
min         0.000000
25%         3.000000
50%         7.000000
75%        14.000000
max      3806.000000
Name: timestamp, dtype: float64

Observations by date in sample:


timestamp
2015-03-20    6873
2015-03-21    3127
Name: count, dtype: int64

### Timestamp and coordinate observations

The sample contains no timestamp or coordinate parsing failures. The longitude and latitude ranges are geographically plausible for Lebanon and nearby areas, with no zero or out-of-range coordinates. There are no exact duplicate rows in the sample.

The first 10,000 rows cover March 20–21, 2015, but this is only the beginning of the file and should not be treated as the dataset's full time range. The median gap between sorted timestamps is 7 seconds, while the maximum gap is 3,806 seconds. This suggests dense observations mixed with pauses or uneven coverage. The repeated timestamps also indicate that multiple vehicles or road observations can occur at the same time, so time alone will not uniquely identify a record.

The next step is a chunked scan of the full raw file. This will measure the actual row count, date range, missingness, coordinate validity, velocity range, and road-segment coverage without loading the entire 540 MB file into memory at once.

## 5. Full-dataset scan

The complete file is scanned in chunks to measure dataset-wide coverage, validity, and summary statistics while keeping memory usage bounded.

In [6]:
chunk_size = 250_000
total_rows = 0
missing_counts = pd.Series(dtype='float64')
date_counts = pd.Series(dtype='float64')
osm_counts = pd.Series(dtype='float64')
timestamp_min = None
timestamp_max = None
invalid_timestamp_rows = 0
invalid_coordinate_rows = 0
velocity_count = 0
velocity_sum = 0.0
velocity_squared_sum = 0.0
velocity_min = float('inf')
velocity_max = float('-inf')

for chunk_number, chunk in enumerate(pd.read_csv(
    RAW_PATH,
    sep=',',
    skipinitialspace=True,
    chunksize=chunk_size,
    low_memory=False
), start=1):
    chunk.columns = (
        chunk.columns
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    total_rows += len(chunk)
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0)

    chunk_timestamp = pd.to_datetime(
        chunk['Date'].astype(str) + ' ' + chunk['Time'].astype(str),
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )
    invalid_timestamp_rows += chunk_timestamp.isna().sum()
    valid_timestamps = chunk_timestamp.dropna()
    if len(valid_timestamps) > 0:
        chunk_min = valid_timestamps.min()
        chunk_max = valid_timestamps.max()
        timestamp_min = chunk_min if timestamp_min is None else min(timestamp_min, chunk_min)
        timestamp_max = chunk_max if timestamp_max is None else max(timestamp_max, chunk_max)
        date_counts = date_counts.add(valid_timestamps.dt.date.value_counts(), fill_value=0)

    coordinate_parts = chunk['Coordinate (Lon, Lat)'].astype('string').str.extract(r'^\s*([^,]+),\s*([^,]+)\s*$')
    longitude = pd.to_numeric(coordinate_parts[0].str.strip(), errors='coerce')
    latitude = pd.to_numeric(coordinate_parts[1].str.strip(), errors='coerce')
    invalid_coordinate_rows += (
        longitude.isna() | latitude.isna() |
        (longitude.abs() > 180) | (latitude.abs() > 90)
    ).sum()

    osm_counts = osm_counts.add(chunk['OSM ID'].value_counts(), fill_value=0)

    velocity = pd.to_numeric(chunk['Velocity'], errors='coerce').dropna()
    velocity_count += len(velocity)
    velocity_sum += velocity.sum()
    velocity_squared_sum += (velocity ** 2).sum()
    if len(velocity) > 0:
        velocity_min = min(velocity_min, velocity.min())
        velocity_max = max(velocity_max, velocity.max())

    if chunk_number % 10 == 0:
        print(f'Processed {total_rows:,} rows...')

velocity_mean = velocity_sum / velocity_count
velocity_variance = (velocity_squared_sum / velocity_count) - (velocity_mean ** 2)

print(f'Total rows: {total_rows:,}')
print(f'Timestamp range: {timestamp_min} to {timestamp_max}')
print(f'Invalid timestamps: {invalid_timestamp_rows:,}')
print(f'Invalid coordinates: {invalid_coordinate_rows:,}')
print(f'Velocity count: {velocity_count:,}')
print(f'Velocity mean: {velocity_mean:,.3f}')
print(f'Velocity standard deviation: {velocity_variance ** 0.5:,.3f}')
print(f'Velocity range: {velocity_min:,.3f} to {velocity_max:,.3f}')
print(f'Unique OSM IDs: {osm_counts.size:,}')
print(f'Missing values by column:\n{missing_counts.astype(int).sort_values(ascending=False)}')

Processed 2,500,000 rows...


Processed 5,000,000 rows...


Total rows: 6,006,401
Timestamp range: 2015-03-20 00:00:48 to 2019-10-17 23:58:41
Invalid timestamps: 0
Invalid coordinates: 0
Velocity count: 6,006,401
Velocity mean: 44.982
Velocity standard deviation: 26.982
Velocity range: 0.000 to 119.999
Unique OSM IDs: 14,289
Missing values by column:
Coordinate (Lon, Lat)    0
Course                   0
Date                     0
OSM ID                   0
Time                     0
Velocity                 0
dtype: int64


In [7]:
print('Rows by date:')
display(date_counts.sort_index().astype(int).to_frame('row_count'))

print('Most frequently observed OSM IDs:')
display(osm_counts.sort_values(ascending=False).head(20).astype(int).to_frame('row_count'))

Rows by date:


,row_count
2015-03-20,6873
2015-03-21,5518
2015-03-22,5676
2015-03-23,5784
2015-03-24,6407
...,...
2019-10-13,1029
2019-10-14,800
2019-10-15,867
2019-10-16,727


Most frequently observed OSM IDs:


,row_count
OSM ID,
way/212368273,66223
way/26547512,64944
way/197464561,49989
way/212368259,47078
way/269505955,43174
way/200723968,41170
way/251916592,38432
way/26547446,36490
way/251916594,35199


### Full-dataset observations

The raw file contains 6,006,401 complete observations covering March 20, 2015 through October 17, 2019. No missing values, invalid timestamps, or invalid coordinate values were found by the full scan. `Velocity` is available for every row, with a dataset-wide range of 0 to approximately 120 and a mean of 44.982. The source documents velocity in meters per second; the zero-velocity rows should be investigated rather than silently removed.

The data spans 1,673 consecutive calendar dates from the first to the last observation. The 14,289 unique `OSM ID` values confirm repeated observations across mapped road segments. The most frequently observed segments account for tens of thousands of rows each, which creates useful repeated structure but also creates a risk of leakage if observations from the same segment or future dates are randomly split between training and test data.

The dataset is technically feasible for historical mobility intelligence. Repeated observations across road segments and time can support road-segment profiles, candidate bottleneck discovery, and unsupervised grouping of recurring movement patterns by behavioral cluster.

## 6. Coverage and duplicate analysis

A complete-looking file can still contain collection gaps or repeated records. These checks quantify the observation calendar and test whether exact duplicates exist across the full dataset.

In [8]:
available_dates = pd.to_datetime(pd.Index(date_counts.index)).sort_values()
full_date_range = pd.date_range(available_dates.min(), available_dates.max(), freq='D')
missing_dates = full_date_range.difference(available_dates)
date_gaps = pd.Series(available_dates).diff().dt.days.dropna()

coverage_summary = pd.Series({
    'first_date': available_dates.min().date(),
    'last_date': available_dates.max().date(),
    'calendar_days_in_range': len(full_date_range),
    'dates_with_observations': len(available_dates),
    'missing_calendar_dates': len(missing_dates),
    'coverage_rate': len(available_dates) / len(full_date_range),
    'largest_gap_between_observed_dates': int(date_gaps.max()),
    'gaps_longer_than_one_day': int((date_gaps > 1).sum())
})

display(coverage_summary.to_frame('value'))
print('First missing dates:')
display(pd.DataFrame({'missing_date': missing_dates[:20].date}))
print('Largest observed-date gaps:')
display(date_gaps.sort_values(ascending=False).head(10).to_frame('gap_days'))

,value
first_date,2015-03-20
last_date,2019-10-17
calendar_days_in_range,1673
dates_with_observations,1673
missing_calendar_dates,0
coverage_rate,1.0
largest_gap_between_observed_dates,1
gaps_longer_than_one_day,0


First missing dates:


,missing_date


Largest observed-date gaps:


,gap_days
1,1.0
2,1.0
3,1.0
4,1.0
5,1.0
6,1.0
7,1.0
8,1.0
9,1.0
10,1.0


In [9]:
duplicate_hashes = []
hourly_group_parts = []
segment_group_parts = []

for chunk_number, chunk in enumerate(pd.read_csv(
    RAW_PATH,
    sep=',',
    skipinitialspace=True,
    chunksize=chunk_size,
    low_memory=False
), start=1):
    chunk.columns = (
        chunk.columns
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    duplicate_hashes.append(
        pd.util.hash_pandas_object(chunk, index=False).to_numpy(dtype='uint64')
    )

    chunk_timestamp = pd.to_datetime(
        chunk['Date'].astype(str) + ' ' + chunk['Time'].astype(str),
        format='%Y-%m-%d %H:%M:%S'
    )
    chunk['date'] = chunk_timestamp.dt.date
    chunk['hour'] = chunk_timestamp.dt.hour
    chunk['day_of_week'] = chunk_timestamp.dt.dayofweek

    hourly_group_parts.append(
        chunk.groupby(['date', 'hour', 'day_of_week'], observed=True)['Velocity']
        .agg(observation_count='count', velocity_sum='sum', velocity_min='min', velocity_max='max')
        .reset_index()
    )
    segment_group_parts.append(
        chunk.groupby('OSM ID', observed=True)['Velocity']
        .agg(observation_count='count', velocity_sum='sum', velocity_min='min', velocity_max='max')
        .reset_index()
    )

all_hashes = np.concatenate(duplicate_hashes)
_, hash_counts = np.unique(all_hashes, return_counts=True)
duplicate_groups = int((hash_counts > 1).sum())
duplicate_rows_beyond_first = int((hash_counts[hash_counts > 1] - 1).sum())

hourly_stats = (
    pd.concat(hourly_group_parts, ignore_index=True)
    .groupby(['date', 'hour', 'day_of_week'], as_index=False)
    .agg(
        observation_count=('observation_count', 'sum'),
        velocity_sum=('velocity_sum', 'sum'),
        velocity_min=('velocity_min', 'min'),
        velocity_max=('velocity_max', 'max')
    )
)
hourly_stats['velocity_mean'] = hourly_stats['velocity_sum'] / hourly_stats['observation_count']

segment_stats = (
    pd.concat(segment_group_parts, ignore_index=True)
    .groupby('OSM ID', as_index=False)
    .agg(
        observation_count=('observation_count', 'sum'),
        velocity_sum=('velocity_sum', 'sum'),
        velocity_min=('velocity_min', 'min'),
        velocity_max=('velocity_max', 'max')
    )
)
segment_stats['velocity_mean'] = segment_stats['velocity_sum'] / segment_stats['observation_count']

print(f'Exact duplicate groups: {duplicate_groups:,}')
print(f'Rows beyond the first in duplicate groups: {duplicate_rows_beyond_first:,}')
print(f'Hourly groups: {len(hourly_stats):,}')
print(f'Segment groups: {len(segment_stats):,}')

Exact duplicate groups: 0
Rows beyond the first in duplicate groups: 0
Hourly groups: 39,623
Segment groups: 14,289


In [10]:
hour_profile = (
    hourly_stats.groupby('hour', as_index=False)
    .agg(observation_count=('observation_count', 'sum'), velocity_sum=('velocity_sum', 'sum'))
)
hour_profile['velocity_mean'] = hour_profile['velocity_sum'] / hour_profile['observation_count']

date_profile = (
    hourly_stats.groupby('date', as_index=False)
    .agg(observation_count=('observation_count', 'sum'), velocity_sum=('velocity_sum', 'sum'))
)
date_profile['velocity_mean'] = date_profile['velocity_sum'] / date_profile['observation_count']

segment_profile = segment_stats.sort_values('observation_count', ascending=False).copy()
top_20_segment_share = segment_profile.head(20)['observation_count'].sum() / total_rows

print('Mean velocity by hour:')
display(hour_profile[['hour', 'observation_count', 'velocity_mean']])
print('Most frequently observed segments:')
display(segment_profile.head(20))
print(f'Top 20 segment share of all observations: {top_20_segment_share:.2%}')
print('Daily mean velocity summary:')
display(date_profile['velocity_mean'].describe().to_frame('daily_mean_velocity'))

Mean velocity by hour:


,hour,observation_count,velocity_mean
0,0,124631,54.784630
1,1,69851,55.787034
2,2,35278,57.453924
3,3,20996,59.739076
4,4,18055,59.876408
5,5,27589,60.255701
6,6,114915,55.461638
7,7,236684,43.905331
8,8,297399,44.219921
9,9,335471,44.186445


Most frequently observed segments:


,OSM ID,observation_count,velocity_sum,velocity_min,velocity_max,velocity_mean
1413,way/212368273,66223,4.282859e+06,0.001238,119.988090,64.673284
3065,way/26547512,64944,2.630510e+06,0.000232,119.999308,40.504280
781,way/197464561,49989,4.078179e+06,0.003878,119.992550,81.581518
1404,way/212368259,47078,1.976187e+06,0.001494,119.793262,41.976866
3661,way/269505955,43174,2.868790e+06,0.002786,119.981952,66.447173
971,way/200723968,41170,1.854101e+06,0.001269,119.832588,45.035246
1894,way/251916592,38432,3.165601e+06,0.001551,119.993897,82.368874
3060,way/26547446,36490,2.166176e+06,0.001055,119.945711,59.363564
1896,way/251916594,35199,1.597412e+06,0.004839,119.991398,45.382308
3101,way/26586371,34445,1.571876e+06,0.002600,118.940022,45.634375


Top 20 segment share of all observations: 12.22%
Daily mean velocity summary:


,daily_mean_velocity
count,1673.000000
mean,45.054367
std,3.306593
min,18.803963
25%,42.723005
50%,44.251321
75%,47.059883
max,56.099957


## 6.5 Zero-Velocity Observations

The full-dataset scan noted that velocity values start at zero, but did not characterize how many zero-velocity rows exist or how they are distributed. Before building velocity-based features in Notebook 01, this question must be closed: are zero-velocity observations concentrated in specific hours or segments, and should they be included in velocity statistics?

This is a quick characterization, not a new sub-project. The raw and cleaned datasets will keep all zero-velocity rows regardless of the outcome.

In [11]:
# --- Zero and near-zero velocity characterization ---
# Targeted check on raw data: inspect exact zeros (== 0) and near-zero (< 0.1 m/s) stationary observations.

zero_exact = 0
near_zero_total = 0  # velocity < 0.1 m/s (~0.36 km/h, stationary)
sub_one_total = 0    # velocity < 1.0 m/s (~3.6 km/h, walking speed / stopped)
near_zero_by_hour = pd.Series(dtype='float64')
near_zero_by_segment = pd.Series(dtype='float64')
min_observed_velocity = float('inf')
scan_total = 0

for chunk in pd.read_csv(
    RAW_PATH,
    sep=',',
    skipinitialspace=True,
    chunksize=chunk_size,
    low_memory=False
):
    chunk.columns = (
        chunk.columns
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    scan_total += len(chunk)

    velocity = pd.to_numeric(chunk['Velocity'], errors='coerce')
    valid_v = velocity.dropna()
    if len(valid_v) > 0:
        min_observed_velocity = min(min_observed_velocity, valid_v.min())

    zero_exact += (velocity == 0).sum()
    is_near_zero = (velocity < 0.1)
    near_zero_total += is_near_zero.sum()
    sub_one_total += (velocity < 1.0).sum()

    if is_near_zero.any():
        chunk_ts = pd.to_datetime(
            chunk.loc[is_near_zero, 'Date'].astype(str) + ' ' +
            chunk.loc[is_near_zero, 'Time'].astype(str),
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        near_zero_by_hour = near_zero_by_hour.add(
            chunk_ts.dt.hour.value_counts(), fill_value=0
        )
        near_zero_by_segment = near_zero_by_segment.add(
            chunk.loc[is_near_zero, 'OSM ID'].value_counts(), fill_value=0
        )

print(f'Total observations scanned: {scan_total:,}')
print(f'Minimum non-null velocity in dataset: {min_observed_velocity:.9e} m/s')
print(f'Exact velocity == 0 observations: {zero_exact:,} (0.00%)')
print(f'Near-zero velocity (< 0.1 m/s / stationary) observations: {near_zero_total:,} ({near_zero_total/scan_total*100:.2f}%)')
print(f'Sub-1.0 m/s (< 3.6 km/h) observations: {sub_one_total:,} ({sub_one_total/scan_total*100:.2f}%)')
print(f'\nSegments with at least one near-zero observation: '
      f'{len(near_zero_by_segment):,} of {osm_counts.size:,} total segments')

# Top segments by near-zero velocity count
top_nz_segments = near_zero_by_segment.sort_values(ascending=False).head(20)
print(f'\nTop 20 segments by near-zero (< 0.1 m/s) velocity count:')
display(pd.DataFrame({
    'near_zero_count': top_nz_segments.astype(int),
    'pct_of_all_near_zeros': (top_nz_segments / near_zero_total * 100).round(2)
}))

# Hour-of-day distribution
nz_by_hour_sorted = near_zero_by_hour.sort_index()
print('\nNear-zero (< 0.1 m/s) observations by hour of day:')
display(pd.DataFrame({
    'near_zero_count': nz_by_hour_sorted.astype(int),
    'pct_of_all_near_zeros': (nz_by_hour_sorted / near_zero_total * 100).round(2)
}))


Total observations scanned: 6,006,401
Minimum non-null velocity in dataset: 2.976993000e-09 m/s
Exact velocity == 0 observations: 0 (0.00%)
Near-zero velocity (< 0.1 m/s / stationary) observations: 43,172 (0.72%)
Sub-1.0 m/s (< 3.6 km/h) observations: 169,125 (2.82%)

Segments with at least one near-zero observation: 4,990 of 14,289 total segments

Top 20 segments by near-zero (< 0.1 m/s) velocity count:


,near_zero_count,pct_of_all_near_zeros
OSM ID,,
way/26655913,384,0.89
way/42860757,375,0.87
way/227026194,344,0.80
way/25893892,330,0.76
way/47281159,288,0.67
way/26547446,229,0.53
way/26307037,203,0.47
way/205362956,195,0.45
way/265398899,193,0.45



Near-zero (< 0.1 m/s) observations by hour of day:


,near_zero_count,pct_of_all_near_zeros
0,828,1.92
1,580,1.34
2,323,0.75
3,215,0.50
4,192,0.44
5,213,0.49
6,571,1.32
7,1689,3.91
8,2018,4.67
9,2191,5.08


### Zero-velocity finding and decision

**Finding:**
1. **Exact zero check:** There are 0 rows with exact mathematical `velocity == 0.0`. The minimum observed velocity is approximately `2.98e-9 m/s` (~0 m/s), which displays as `0.000` in standard summary tables due to floating-point representation of stationary GPS fixes.
2. **Stationary / near-zero observations:** There are 43,172 observations (0.72% of the dataset) with velocity under 0.1 m/s (~0.36 km/h), and 169,125 observations (2.82%) under 1.0 m/s (~3.6 km/h).
3. **Distribution:** Near-zero observations are broadly distributed across hours of the day (~0.5%–1.0% of observations in any given hour) and across thousands of road segments.

**Decision (applied consistently across Notebooks 00 and 01):**

- **Keep** all observations in raw and cleaned datasets. They represent legitimate crowdsourced GPS observations.
- **Exclude** zero and near-zero (< 0.1 m/s) observations from velocity-based feature calculations (median velocity, velocity standard deviation, and relative slowdowns). Stationary/stop events without duration or vehicle trajectory context should not artificially distort movement-velocity baselines.
- **Preserve** the count of zero/near-zero observations per segment as a diagnostic field (`zero_velocity_observation_count`, counting observations < 0.1 m/s) in the `segment_features` table, ensuring no information is lost.


## 7. Feasibility decision

The dataset is suitable for a focused AI for Lebanon project on historical mobility intelligence. It contains repeated, geolocated observations across mapped road segments and time, which allows us to identify candidate recurring bottlenecks and group road segments by movement behavior.

The primary analytical object is a road-segment profile: observed velocity, variation, observation count, and time-of-day behavior. Machine learning supports this through unsupervised clustering. SQL and visual analysis establish the patterns before any model is trusted.

**Zero-velocity closure:** Zero and near-zero observations have been quantified and characterized above. They are kept in the cleaned dataset but excluded from velocity-based feature calculations, with their count preserved as a diagnostic field. This decision is applied consistently in Notebook 01.

**What the dataset supports:** segment-level and time-level historical pattern discovery, candidate bottleneck screening, unsupervised grouping of road segments by recurring movement behavior, and decision-support ranking for transportation planners.

**What the dataset does not support:** live congestion measurement, causal explanations for slowdowns, accident prediction, traffic volume estimation, road capacity analysis, intervention evaluation, or any claim that requires speed limits, vehicle identifiers, route trajectories, weather, incident logs, or real-time coverage. The output is a decision-support prototype that highlights candidate locations and time windows for further investigation, planning, or future data collection.

## 8. Export cleaned analysis data

The processed file keeps the raw observations traceable while adding reusable temporal and coordinate fields for SQL, visualization, and later machine-learning work. It is written in chunks so the full cleaned dataset does not need to remain in memory.

In [12]:
PROCESSED_FILE = PROCESSED_DIR / 'traffic_clean.csv'
write_header = True
processed_rows = 0

for chunk in pd.read_csv(
    RAW_PATH,
    sep=',',
    skipinitialspace=True,
    chunksize=chunk_size,
    low_memory=False
):
    chunk.columns = (
        chunk.columns
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    timestamp = pd.to_datetime(
        chunk['Date'].astype(str) + ' ' + chunk['Time'].astype(str),
        format='%Y-%m-%d %H:%M:%S'
    )
    coordinate_parts = chunk['Coordinate (Lon, Lat)'].astype('string').str.extract(r'^\s*([^,]+),\s*([^,]+)\s*$')

    clean_chunk = pd.DataFrame({
        'timestamp': timestamp,
        'date': timestamp.dt.date.astype('string'),
        'year': timestamp.dt.year.astype('int16'),
        'month': timestamp.dt.month.astype('int8'),
        'day_of_week': timestamp.dt.dayofweek.astype('int8'),
        'hour': timestamp.dt.hour.astype('int8'),
        'longitude': pd.to_numeric(coordinate_parts[0].str.strip(), errors='coerce'),
        'latitude': pd.to_numeric(coordinate_parts[1].str.strip(), errors='coerce'),
        'course': pd.to_numeric(chunk['Course'], errors='coerce'),
        'velocity': pd.to_numeric(chunk['Velocity'], errors='coerce'),
        'osm_id': chunk['OSM ID'].astype('string')
    })

    clean_chunk.to_csv(
        PROCESSED_FILE,
        mode='w' if write_header else 'a',
        header=write_header,
        index=False
    )
    write_header = False
    processed_rows += len(clean_chunk)

print(f'Exported rows: {processed_rows:,}')
print(f'Processed file: {PROCESSED_FILE}')
print(f'Processed file size: {PROCESSED_FILE.stat().st_size / 1024**2:,.1f} MB')

Exported rows: 6,006,401
Processed file: ..\data\processed\traffic_clean.csv
Processed file size: 581.5 MB


## 9. Next notebook

Notebook 01 loads `traffic_clean.csv` into SQLite and builds road-segment and time-based profiles, culminating in a `segment_features` table. Notebook 02 groups road segments by recurring movement behavior using unsupervised K-Means clustering. Notebook 03 presents candidate bottleneck maps, temporal patterns, practical use cases, and limitations.